# Consulta CNPJ — Receita Federal
Rode a Célula 1 sempre primeiro. Depois escolha o que precisa.

In [ ]:
# CÉLULA 1 — Configuração (rode sempre primeiro)
import psycopg2
import pandas as pd
import warnings
import os
from datetime import date
warnings.filterwarnings('ignore')

PG_HOST     = 'localhost'
PG_PORT     = 5432
PG_USER     = 'postgres'
PG_PASSWORD = 'Amorauva25@'
PG_DATABASE = 'rfb_cnpj'

def conn():
    return psycopg2.connect(
        host=PG_HOST, port=PG_PORT,
        user=PG_USER, password=PG_PASSWORD,
        database=PG_DATABASE, connect_timeout=10
    )

# Verifica totais no banco
c = conn()
cur = c.cursor()
print('BANCO rfb_cnpj — TOTAIS:')
print('-'*35)
for t in ['empresas','estabelecimentos','socios','cnaes','municipios']:
    cur.execute(f'SELECT COUNT(*) FROM {t}')
    print(f'  {t:<22} {cur.fetchone()[0]:>12,}')
print('-'*35)
cur.close()
c.close()
print('Pronto para consultas!')

In [ ]:
# CÉLULA 2 — Ver CNAEs disponíveis (pesquise por palavra-chave)
BUSCA = 'medic'  # <-- altere aqui: medic, odonto, farmac, autopc, perfum, etc

c = conn()
cur = c.cursor()
cur.execute(
    'SELECT codigo, descricao FROM cnaes WHERE descricao ILIKE %s ORDER BY codigo',
    (f'%{BUSCA}%',)
)
rows = cur.fetchall()
cur.close()
c.close()
print(f'{len(rows)} CNAEs encontrados para "{BUSCA}":')
for r in rows:
    print(f'  {r[0]} -> {r[1]}')

In [ ]:
# CÉLULA 3 — Ver municípios por nome
BUSCA_MUN = 'recife'  # <-- altere aqui

c = conn()
cur = c.cursor()
cur.execute(
    'SELECT codigo, nome FROM municipios WHERE nome ILIKE %s ORDER BY nome',
    (f'%{BUSCA_MUN}%',)
)
rows = cur.fetchall()
cur.close()
c.close()
print(f'{len(rows)} município(s) encontrado(s):')
for r in rows:
    print(f'  {r[0]} -> {r[1]}')

In [ ]:
# CÉLULA 4 — PROSPECÇÃO (altere os filtros)
# ============================================
CNAES_ALVO = ('8630501','8630502','8630503','8630504','8630599')  # Clinicas medicas
UFS_ALVO   = ('PE','CE','GO','MT')  # Estados
SITUACAO   = '02'  # 02=Ativa
# ============================================

c = conn()
df = pd.read_sql(f'''
SELECT
    est.cnpj_basico || est.cnpj_ordem || est.cnpj_dv AS cnpj,
    COALESCE(NULLIF(est.nome_fantasia,''), emp.razao_social) AS nome,
    emp.razao_social,
    est.cnae_fiscal,
    mu.nome       AS municipio,
    est.uf,
    CASE WHEN est.ddd1 <> '' THEN '(' || est.ddd1 || ') ' || est.telefone1 ELSE '' END AS telefone1,
    CASE WHEN est.ddd2 <> '' THEN '(' || est.ddd2 || ') ' || est.telefone2 ELSE '' END AS telefone2,
    est.email,
    s.nome_socio,
    s.qualificacao
FROM estabelecimentos est
LEFT JOIN empresas   emp ON emp.cnpj_basico = est.cnpj_basico
LEFT JOIN municipios mu  ON mu.codigo       = est.cod_municipio
LEFT JOIN socios     s   ON s.cnpj_basico   = est.cnpj_basico
WHERE est.cnae_fiscal IN {CNAES_ALVO}
  AND est.uf          IN {UFS_ALVO}
  AND est.situacao    = '{SITUACAO}'
ORDER BY est.uf, mu.nome
''', c)
c.close()
print(f'{len(df):,} registros encontrados!')
df.head(20)

In [ ]:
# CÉLULA 5 — Exportar resultado para Excel
saida = rf'C:\Users\nayar\OneDrive\Documentos\prospeccao_{date.today()}.xlsx'
df.to_excel(saida, index=False)
print(f'{len(df):,} registros exportados!')
print(f'Arquivo: {saida}')

In [ ]:
# CÉLULA 6 — Contagem por UF e CNAE (visão geral)
CNAES_ALVO = ('8630501','8630502','8630503','8630504','8630599')

c = conn()
cur = c.cursor()
cur.execute(f'''
    SELECT uf, cnae_fiscal, COUNT(*) as total
    FROM estabelecimentos
    WHERE cnae_fiscal IN {CNAES_ALVO}
    AND situacao = '02'
    GROUP BY uf, cnae_fiscal
    ORDER BY total DESC
    LIMIT 30
''')
print('UF | CNAE | Total')
print('-'*30)
for r in cur.fetchall():
    print(f'  {r[0]:<5} {r[1]:<10} {r[2]:>8,}')
cur.close()
c.close()

In [ ]:
# CÉLULA 7 — Buscar empresa por nome
BUSCA_NOME = 'clinica'  # <-- altere aqui

c = conn()
df_nome = pd.read_sql(f'''
SELECT
    est.cnpj_basico || est.cnpj_ordem || est.cnpj_dv AS cnpj,
    COALESCE(NULLIF(est.nome_fantasia,''), emp.razao_social) AS nome,
    est.cnae_fiscal,
    mu.nome AS municipio,
    est.uf,
    CASE WHEN est.ddd1 <> '' THEN '(' || est.ddd1 || ') ' || est.telefone1 ELSE '' END AS telefone,
    est.email
FROM estabelecimentos est
LEFT JOIN empresas   emp ON emp.cnpj_basico = est.cnpj_basico
LEFT JOIN municipios mu  ON mu.codigo       = est.cod_municipio
WHERE (est.nome_fantasia ILIKE '%{BUSCA_NOME}%'
   OR  emp.razao_social  ILIKE '%{BUSCA_NOME}%')
AND est.situacao = '02'
LIMIT 100
''', c)
c.close()
print(f'{len(df_nome):,} resultado(s)')
df_nome.head(20)

In [ ]:
# CÉLULA 8 — Buscar sócios de uma empresa
CNPJ_BASICO = '12345678'  # <-- coloque os 8 primeiros dígitos do CNPJ

c = conn()
cur = c.cursor()
cur.execute('''
    SELECT s.nome_socio, s.qualificacao, s.data_entrada,
           e.razao_social
    FROM socios s
    JOIN empresas e ON e.cnpj_basico = s.cnpj_basico
    WHERE s.cnpj_basico = %s
''', (CNPJ_BASICO,))
rows = cur.fetchall()
cur.close()
c.close()
print(f'{len(rows)} socio(s) encontrado(s):')
for r in rows:
    print(f'  Nome: {r[0]}')
    print(f'  Qual: {r[1]}')
    print(f'  Desde: {r[2]}')
    print(f'  Empresa: {r[3]}')
    print()